# Human-in-the-Loop in AutoGen

Sometimes you want a human to step in — to approve, correct, or add information mid-conversation. AutoGen handles this with **`UserProxyAgent`**: an agent that, instead of calling an LLM, asks *you* what to say.

Two common patterns:
1. **Approval flow** — assistant proposes, human approves or rejects.
2. **Custom input function** — plug in your own prompt source (web UI, Slack, etc.) instead of `input()`.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

## 1. Approval flow with `UserProxyAgent`

The assistant drafts a reply, then the user proxy asks **you** to type something. The team stops when you type `APPROVE`.

In [ ]:
writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message="Write a one-sentence answer to the user's question.",
)

human = UserProxyAgent(name="human")

team = RoundRobinGroupChat(
    [writer, human],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(6),
)

await Console(team.run_stream(task="What is an AI agent?"))

When the cell pauses, look in the kernel's input prompt and type either feedback (e.g. *make it shorter*) or `APPROVE` to end.

## 2. Custom `input_func`

Instead of reading from stdin, supply your own coroutine. This is how you'd wire human input from a web UI, Slack message, or hard-coded script.

In [ ]:
scripted_replies = iter(["make it shorter", "APPROVE"])

async def my_input(prompt: str, cancellation_token=None) -> str:
    reply = next(scripted_replies)
    print(f"  [human-bot] -> {reply}")
    return reply

human2 = UserProxyAgent(name="human", input_func=my_input)

team2 = RoundRobinGroupChat(
    [writer, human2],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(6),
)

await Console(team2.run_stream(task="What is RAG?"))

## Recap

- **`UserProxyAgent`** plugs a human into any team like another agent.
- Default: it reads from stdin via `input()`.
- Pass `input_func=...` to source replies from anywhere — script, queue, web UI.
- Pair it with `TextMentionTermination("APPROVE")` for clean approval flows.

In [ ]:
await model_client.close()